In [2]:
# =====================================================================================
# |                                 PURPOSE
# -------------------------------------------------------------------------------------
# Convert the SQL database into CSV files stored inside "barbell" and "none" folders.
# Each session = 1 CSV example.
# =====================================================================================

import os
import sqlite3
import pandas as pd

# -------------------------
# Parameters
# -------------------------
db_path = "sensor_database.db"
axes = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

barbell_type = "BARBELL_CARRYING"
noise_type = "DEVICE_CARRYING"

output_root = "./raw_data"
barbell_dir = os.path.join(output_root, "barbell")
noise_dir = os.path.join(output_root, "none")

os.makedirs(barbell_dir, exist_ok=True)
os.makedirs(noise_dir, exist_ok=True)

# -------------------------
# Fetch all sessions from DB by type
# -------------------------
def get_all_sessions_data(db_path, label_type):
    conn = sqlite3.connect(db_path)
    
    session_ids = pd.read_sql(
        f"SELECT sessionId FROM session WHERE noise = '{label_type}' ORDER BY sessionId ASC;",
        conn
    )["sessionId"].tolist()

    sessions_data = {}
    for sid in session_ids:
        df = pd.read_sql(
            f"SELECT ax, ay, az, gx, gy, gz FROM rawdata WHERE sessionId = {sid}",
            conn
        )
        sessions_data[sid] = df
    
    conn.close()
    return sessions_data

# -------------------------
# Load all sessions
# -------------------------
barbell_sessions = get_all_sessions_data(db_path, barbell_type)
noise_sessions = get_all_sessions_data(db_path, noise_type)

print(f"Barbell sessions: {len(barbell_sessions)}")
print(f"Noise sessions: {len(noise_sessions)}")

# -------------------------
# Save each session as CSV
# -------------------------
def save_sessions_to_csv(sessions_dict, target_folder, prefix):
    for i, (session_id, df) in enumerate(sessions_dict.items()):
        filename = f"{prefix}_{session_id}.csv"
        path = os.path.join(target_folder, filename)
        df.to_csv(path, index=False)
        print("Saved:", path)

# Save barbell examples
save_sessions_to_csv(barbell_sessions, barbell_dir, "barbell")

# Save "none" examples
save_sessions_to_csv(noise_sessions, noise_dir, "none")

print("Done exporting CSV dataset!")

Barbell sessions: 13
Noise sessions: 27
Saved: ./raw_data/barbell/barbell_52.csv
Saved: ./raw_data/barbell/barbell_53.csv
Saved: ./raw_data/barbell/barbell_54.csv
Saved: ./raw_data/barbell/barbell_55.csv
Saved: ./raw_data/barbell/barbell_56.csv
Saved: ./raw_data/barbell/barbell_57.csv
Saved: ./raw_data/barbell/barbell_58.csv
Saved: ./raw_data/barbell/barbell_59.csv
Saved: ./raw_data/barbell/barbell_60.csv
Saved: ./raw_data/barbell/barbell_61.csv
Saved: ./raw_data/barbell/barbell_62.csv
Saved: ./raw_data/barbell/barbell_63.csv
Saved: ./raw_data/barbell/barbell_64.csv
Saved: ./raw_data/none/none_1.csv
Saved: ./raw_data/none/none_2.csv
Saved: ./raw_data/none/none_3.csv
Saved: ./raw_data/none/none_4.csv
Saved: ./raw_data/none/none_5.csv
Saved: ./raw_data/none/none_6.csv
Saved: ./raw_data/none/none_7.csv
Saved: ./raw_data/none/none_8.csv
Saved: ./raw_data/none/none_9.csv
Saved: ./raw_data/none/none_10.csv
Saved: ./raw_data/none/none_11.csv
Saved: ./raw_data/none/none_12.csv
Saved: ./raw_dat

# Save as FFT (only the gy parameter)

In [15]:
import os
import sqlite3
import pandas as pd
import numpy as np
from thinkdsp import Wave

# -------------------------
# Parameters
# -------------------------
db_path = "sensor_database.db"
fs = 125  # sensor sampling rate
axes = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

barbell_type = "BARBELL_CARRYING"
barbell_type2 = "BARBELL_STATIONARY"
noise_type = "DEVICE_CARRYING"
noise_type2 = "DEVICE_STATIONARY"

output_root = "./raw_data"
barbell_dir = os.path.join(output_root, "barbell")
noise_dir = os.path.join(output_root, "none")

os.makedirs(barbell_dir, exist_ok=True)
os.makedirs(noise_dir, exist_ok=True)

# -------------------------
# Get Count
# -------------------------
def balance_count_of_classes(db_path, label_type, label_type2):
    conn = sqlite3.connect(db_path)
    
    count1 = pd.read_sql(
        f"SELECT COUNT(sessionId) FROM session WHERE noise = '{label_type}'",
        conn
    ).iloc[0,0] # extract int from dataframe
    
    count2 = pd.read_sql(
        f"SELECT COUNT(sessionId) FROM session WHERE noise = '{label_type2}'",
        conn
    ).iloc[0,0]
    return count1 if count1 <= count2 else count2

# -------------------------
# Fetch all sessions from DB by type
# -------------------------
def get_all_sessions_data(db_path, label_type, label_type2, count1, count2):
    conn = sqlite3.connect(db_path)

    session_ids = pd.read_sql(
        f"SELECT sessionId FROM session WHERE noise = '{label_type}' ORDER BY sessionId ASC LIMIT {count1};",
        conn
    )["sessionId"].tolist()
    
    session_ids.extend( pd.read_sql(
        f"SELECT sessionId FROM session WHERE noise = '{label_type2}' ORDER BY sessionId ASC LIMIT {count2};",
        conn
    )["sessionId"].tolist())

    sessions_data = {}
    for sid in session_ids:
        df = pd.read_sql(
            f"SELECT ax, ay, az, gx, gy, gz FROM rawdata WHERE sessionId = {sid}",
            conn
        )
        sessions_data[sid] = df
    
    conn.close()
    return sessions_data

# -------------------------
# Compute FFT and save as CSV
# -------------------------
def save_fft_sessions(sessions_dict, target_folder, prefix, axis='gy'):
    for session_id, df in sessions_dict.items():
        ys = df[axis].values
        wave = Wave(ys, framerate=fs)
        spectrum = wave.make_spectrum()
        
        # Save frequency and amplitude
        fft_df = pd.DataFrame({
            'frequency': spectrum.fs,
            'amplitude': spectrum.amps
        })
        
        filename = f"{prefix}_{session_id}.csv"
        path = os.path.join(target_folder, filename)
        fft_df.to_csv(path, index=False)
        print("Saved FFT CSV:", path)

# -------------------------
# Load sessions
# -------------------------
balance_count_of_type1 = balance_count_of_classes(db_path, barbell_type, noise_type)
balance_count_of_type2 = balance_count_of_classes(db_path, barbell_type2, noise_type2)

barbell_sessions = get_all_sessions_data(db_path, barbell_type, barbell_type2, balance_count_of_type1, balance_count_of_type2)
noise_sessions = get_all_sessions_data(db_path, noise_type, noise_type2, balance_count_of_type1, balance_count_of_type2)

print(f"Barbell sessions: {len(barbell_sessions)}")
print(f"Noise sessions: {len(noise_sessions)}")

# -------------------------
# Save FFT CSVs
# -------------------------
save_fft_sessions(barbell_sessions, barbell_dir, "barbell", axis='gy')
save_fft_sessions(noise_sessions, noise_dir, "none", axis='gy')

print("Done exporting FFT dataset!")


Barbell sessions: 29
Noise sessions: 29
Saved FFT CSV: ./raw_data/barbell/barbell_52.csv
Saved FFT CSV: ./raw_data/barbell/barbell_53.csv
Saved FFT CSV: ./raw_data/barbell/barbell_54.csv
Saved FFT CSV: ./raw_data/barbell/barbell_55.csv
Saved FFT CSV: ./raw_data/barbell/barbell_56.csv
Saved FFT CSV: ./raw_data/barbell/barbell_57.csv
Saved FFT CSV: ./raw_data/barbell/barbell_58.csv
Saved FFT CSV: ./raw_data/barbell/barbell_59.csv
Saved FFT CSV: ./raw_data/barbell/barbell_60.csv
Saved FFT CSV: ./raw_data/barbell/barbell_61.csv
Saved FFT CSV: ./raw_data/barbell/barbell_62.csv
Saved FFT CSV: ./raw_data/barbell/barbell_63.csv
Saved FFT CSV: ./raw_data/barbell/barbell_64.csv
Saved FFT CSV: ./raw_data/barbell/barbell_43.csv
Saved FFT CSV: ./raw_data/barbell/barbell_44.csv
Saved FFT CSV: ./raw_data/barbell/barbell_45.csv
Saved FFT CSV: ./raw_data/barbell/barbell_46.csv
Saved FFT CSV: ./raw_data/barbell/barbell_47.csv
Saved FFT CSV: ./raw_data/barbell/barbell_48.csv
Saved FFT CSV: ./raw_data/bar